# 3. The flag line and the errors

## What you will learn

- What the "flag line" is (the top 10% of an earlier year's scores), and why it comes from an earlier year.
- How to read the four outcomes of a flag: caught downgrades, missed downgrades, false alarms, and quiet and correct
  cases (not flagged, not downgraded).
- Why a flag is a reason to look at a company, not a reason to sell it.

## What you need

- Notebooks 1 and 2 (the panel, the label, the year-by-year test, ROC-AUC).
- The rebuild from notebook 5. It writes the real table `OUT_DIR/panel_v2.csv` and the test results in `OUT_DIR/run/`.
  This notebook only reads those files; it does not download or rebuild anything.
- Python with the packages in `model/requirements.txt` and `jupyter`.

> **Keep licensed data to yourself.** On the WRDS route the table and every result are licensed: keep them on your own
> computer and clear the outputs of this notebook before you share it.

## Set up

The first cell only sets the folders. `DATA_DIR` is where downloads and hand-downloaded files live; `OUT_DIR` is where
the rebuild wrote its table and results. Both can be changed. The notebook also reads the two environment variables
`TCF_NB_DATA_DIR` and `TCF_NB_OUT_DIR` if they are set (the automatic test uses them), so you can point it at another
folder.

In [ ]:
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd

# Find the repository folder (the one that holds model/src/tcf_credit). This works whether the notebook
# is started from the repository folder or from model/notebooks.
here = Path.cwd().resolve()
root = next(p for p in [here, *here.parents] if (p / "model" / "src" / "tcf_credit").is_dir())
SRC = root / "model" / "src"
sys.path[:0] = [str(SRC)]
old_path = os.environ.get("PYTHONPATH")
os.environ["PYTHONPATH"] = os.pathsep.join([str(SRC)] + ([old_path] if old_path else []))

DATA_DIR = Path(os.environ.get("TCF_NB_DATA_DIR", root / "data"))
OUT_DIR = Path(os.environ.get("TCF_NB_OUT_DIR", root / "build"))

pd.set_option("display.width", 150)
pd.set_option("display.max_columns", 40)
print("repository:", root)
print("data folder:", DATA_DIR)
print("output folder:", OUT_DIR)


In [ ]:
panel_path = OUT_DIR / "panel_v2.csv"
run_dir = OUT_DIR / "run"
needed = [panel_path, run_dir / "pooled.csv", run_dir / "per_year.csv",
          run_dir / "predictions.csv", run_dir / "params.json"]
ready = all(path.exists() for path in needed)
if not panel_path.exists():
    print("No real table yet: run the rebuild first (see notebook 5).")
elif not ready:
    print("The rebuild's results are not in OUT_DIR/run yet: run the rebuild first (see notebook 5).")
else:
    print("Found the real table and the rebuild's results in", OUT_DIR)


## The rebuild's results

The rebuild's step 8 tests the models year by year and writes four files in `OUT_DIR/run/`:

- `predictions.csv`: every test score, with the two flags of each report (`flag_cut` and `flag_rank`).
- `pooled.csv`: all test years together, one line per model and flag rule.
- `per_year.csv`: the same counts for each test year.
- `params.json`: the chosen settings, the inputs and the models of the run.

The next cell reads the table and the results. The notebook uses the logit, the model the lecture reports.

In [ ]:
if not ready:
    print("No real table yet: run the rebuild first (see notebook 5).")
else:
    panel = pd.read_csv(panel_path)
    pred = pd.read_csv(run_dir / "predictions.csv")
    pooled = pd.read_csv(run_dir / "pooled.csv")
    lg = pred[pred.model == "logit"]
    print(len(lg), "test scores for the logit, for test years", int(lg.year.min()), "to", int(lg.year.max()))
    display(lg.head(4))


## What is a flag, and where does the line come from?

A probability is not yet a decision. To act, we need a **flag line**: a score above which we say "look at this
company". The model uses the following rule, set once in `evaluate.py`:

> The flag line for test year **Y** is the score that only the top **10%** of reports of year **Y-2** exceed. Those
> reports are scored by a model trained on years up to **Y-4**, so they are scored "out of sample" (by a model that
> never saw them), the way the test year will be. A report in year Y is **flagged** if its probability is at or above
> that line.

**Why from an earlier year and not from year Y itself?** If we took the top 10% of year Y's own scores, we would need
to know all of year Y's scores before flagging any single company. In real life you receive the reports one at a time
during the year. Fixing the line in advance, from a year that is already finished, is something you could really do. A
side effect: the share flagged is not exactly 10% each year. It goes up when the scores are higher than in the earlier
year and down when they are lower (on the lecture's real data it was 6.6% in 2020). (The rebuild also writes a second,
"rank" rule that flags about 10% of each test year. We keep it only to compare.)

The next cell rebuilds the flag line of one test year with the package's own functions and checks it against the
rebuild's output.

In [ ]:
if not ready:
    print("No real table yet: run the rebuild first (see notebook 5).")
else:
    import json
    from tcf_credit import evaluate as E

    run_params = json.loads((run_dir / "params.json").read_text())
    E.DROP[:] = run_params.get("drop", [])
    E.EXTRA[:] = run_params.get("extra", [])
    E.ONLY[:] = list(run_params.get("chosen", {}))
    data = pd.read_csv(panel_path)
    data = E.fixed_order(data)        # the same row order as the rebuild's evaluate step, so the fits match
    data = data[data.has_prices == 1].copy()
    data["year"] = pd.to_datetime(data["t"]).dt.year
    data["ebit_interest"] = data["ebit_interest"].clip(-50, 100)

    make, cols_, impute, sectors = E.models(run_params)["logit"]

    years_available = sorted(int(y) for y in lg.year.unique())
    Y = 2020 if 2020 in years_available else years_available[-1]
    train, test = data[data.year <= Y - 2], data[data.year == Y]
    calib_train, calib = data[data.year <= Y - 4], data[data.year == Y - 2]   # the earlier year that sets the line

    _, p_test = E.fit_predict(make, cols_, impute, sectors, train, test)          # model for year Y, scores year Y
    _, p_calib = E.fit_predict(make, cols_, impute, sectors, calib_train, calib)  # older model, scores year Y-2
    line = np.quantile(p_calib, 1 - 0.10)
    flagged = p_test >= line

    print(f"test year {Y}: the line is {line:.3f}; it is the score that 10% of the {len(calib)} reports of {Y-2} "
          "reach or exceed.")
    print(f"{flagged.sum()} of {len(test)} reports in {Y} are flagged ({flagged.mean():.1%}).")

    # A check that can fail: the same flags as the rebuild wrote.
    cmd = pred[(pred.model == "logit") & (pred.year == Y)].reset_index(drop=True)
    assert len(cmd) == len(flagged), "the rebuild's scores and this check cover different reports"
    assert (cmd.flag_cut.to_numpy() == flagged).all(), "the by-hand flags differ from the rebuild's flags"
    print("same flags as predictions.csv: yes")


## Flags year by year

The table shows, for each test year, the share of reports flagged by the fixed line ("earlier-year line") and by the
rank rule (about 10% of each year).

In [ ]:
if not ready:
    print("No real table yet: run the rebuild first (see notebook 5).")
else:
    lg = pred[pred.model == "logit"]
    by_year = lg.groupby("year").agg(reports=("label", "size"), downgrades=("label", "sum"),
                                     flagged_by_line=("flag_cut", "mean"),
                                     flagged_by_rank=("flag_rank", "mean")).round(3)
    display(by_year)


## Caught, missed, false alarms: the 2x2 table

Every test report ends in one of four boxes:

|  | Downgraded within 12 months | Not downgraded |
|---|---|---|
| **Flagged** | caught (true positive) | false alarm (false positive) |
| **Not flagged** | missed (false negative) | quiet and correct (true negative) |

Two rates matter and they have different denominators:

- **Share of downgrades caught** = caught / (caught + missed). It is about the downgrades.
- **False-alarm rate** = false alarms / (all reports that were not downgraded). It is about the companies that did
  fine.

In [ ]:
if not ready:
    print("No real table yet: run the rebuild first (see notebook 5).")
else:
    c = pooled[(pooled.model == "logit") & (pooled.rule == "calibrated_cut")].iloc[0]
    table = pd.DataFrame([[c.TP, c.FP], [c.FN, c.TN]],
                         index=["Flagged", "Not flagged"],
                         columns=["Downgraded within 12 months", "Not downgraded"]).astype(int)
    table.loc["Total"] = table.sum()
    table["Total"] = table.sum(axis=1)
    print(table.to_string())
    print()
    print(f"share of downgrades caught: {c.TP}/{c.TP + c.FN} = {c.TP / (c.TP + c.FN):.1%}   "
          f"(missed: {c.miss_rate:.1%})")
    print(f"false-alarm rate: {c.FP}/{c.FP + c.TN} = {c.false_alarm_rate:.1%}")
    print(f"share of flags that were real downgrades (precision): {c.TP}/{c.flagged} = {c.precision:.1%}; "
          f"share of all reports that were downgraded: {c.base_rate:.1%}")


These are your numbers, from the files the rebuild wrote. For the lecture's real data, the flag line caught **407 of
the 862** downgrades (so 455 were missed) with **8.7%** false alarms. Reading that: the line finds about half of the
downgrades in advance, and about 9 in 100 healthy companies get flagged anyway. The model is an early-warning light,
not a crystal ball. The free route and the WRDS route give different tables, so your numbers need not match the
lecture's.

Moving the line trades the two errors against each other. Flag more companies and you catch more downgrades but raise
more false alarms.

**Two flag rules, side by side.** `predictions.csv` holds two flags for every report, and they give different counts
on purpose:

| Rule | How a report is flagged | Used for |
|---|---|---|
| Earlier-year line (`flag_cut`) | score at or above an earlier year's line | lecture's results; usable in real time |
| Rank rule (`flag_rank`) | among the riskiest share of the same test year | only to compare; needs the whole year |

The share flagged by the earlier-year line changes from year to year; the rank rule flags about the same share each
year.

So do not expect the "10%" row below to equal the 2x2 table above. The first row of the next table repeats the
earlier-year line so that you can compare the two directly.

In [ ]:
if not ready:
    print("No real table yet: run the rebuild first (see notebook 5).")
else:
    n_reports, n_down = len(lg), int(lg.label.sum())
    rows = []
    # first row: the earlier-year line (the 2x2 table above)
    tp = int((lg.flag_cut & (lg.label == 1)).sum()); fp = int((lg.flag_cut & (lg.label == 0)).sum())
    rows.append({"rule": "earlier-year line", "share flagged": f"{lg.flag_cut.mean():.1%}", "caught": tp,
                 "missed": n_down - tp, "share of downgrades caught": round(tp / n_down, 3), "false alarms": fp,
                 "false-alarm rate": round(fp / (n_reports - n_down), 3)})
    for share in [0.05, 0.10, 0.20, 0.30]:
        flag = lg.groupby("year").p.transform(lambda s: s >= np.quantile(s, 1 - share))
        tp = int((flag & (lg.label == 1)).sum()); fn = int((~flag & (lg.label == 1)).sum())
        fp = int((flag & (lg.label == 0)).sum()); tn = int((~flag & (lg.label == 0)).sum())
        rows.append({"rule": "rank rule", "share flagged": f"{share:.0%}", "caught": tp, "missed": fn,
                     "share of downgrades caught": round(tp / (tp + fn), 3), "false alarms": fp,
                     "false-alarm rate": round(fp / (fp + tn), 3)})
    display(pd.DataFrame(rows).set_index(["rule", "share flagged"]))


## The level of the probabilities in a shock year

A model can **rank** companies well and still get the **level** of the probabilities wrong in a year when something
hits everybody at once. On the lecture's real data, in 2020 (the pandemic year) about **29%** of the reports were
followed by a downgrade, while the model's average predicted probability was only about **14%**. The ranking can still
be good (that is what ROC-AUC measures), but the probabilities were too low for that year. The flag line for 2020 was
fixed from the scores of 2018, when the model's average probability was about 23%. Against that line the low 2020
probabilities rarely reached it: the line flagged only 46 of the 695 reports (6.6%), fewer than the 10% it was set
for, and caught only 39 of the 203 downgrades. Probabilities that are too low put fewer companies above a line fixed
earlier, not more.

The table compares, for each test year of your table, the actual downgrade rate with the average predicted
probability, and shows the ROC-AUC and the share flagged. In an ordinary year the first two columns should stay
fairly close; check whether they do. A year like 2020 in the lecture, when the economy moves suddenly, can pull
them apart even when the ranking stays good.

In [ ]:
if not ready:
    print("No real table yet: run the rebuild first (see notebook 5).")
else:
    from sklearn.metrics import roc_auc_score
    level = lg.groupby("year").apply(lambda g: pd.Series({
        "actual downgrade rate": g.label.mean(),
        "average predicted probability": g.p.mean(),
        "ROC-AUC": roc_auc_score(g.label, g.p),
        "share flagged (earlier-year line)": g.flag_cut.mean()}), include_groups=False).round(3)
    display(level)


## Look at the errors

Numbers hide the cases. Here are the real companies behind the errors in your table: the downgrades the model
**missed** (lowest scores among downgraded reports) and the **false alarms** (highest scores among reports without a
downgrade). The rating state is shown next to each one, so you can see whether the rating had already moved (a
negative outlook or CreditWatch) before the event. Our reading, not tested: a model that leans on the rating has
trouble with companies whose rating had not yet moved.

In [ ]:
if not ready:
    print("No real table yet: run the rebuild first (see notebook 5).")
else:
    panel = pd.read_csv(panel_path)
    look = lg.merge(panel[["cik", "t", "sp_name", "rating", "outlook_neg", "watch_neg", "debt_assets",
                           "ebit_assets"]], on=["cik", "t"], how="left")
    show = ["sp_name", "t", "p", "rating", "outlook_neg", "watch_neg", "debt_assets", "ebit_assets"]
    missed = look[(look.label == 1) & (~look.flag_cut)].sort_values("p").head(5)
    alarms = look[(look.label == 0) & (look.flag_cut)].sort_values("p", ascending=False).head(5)
    print("Downgrades the model missed (lowest scores):")
    display(missed[show].round(3).reset_index(drop=True))
    print("False alarms (highest scores without a downgrade):")
    display(alarms[show].round(3).reset_index(drop=True))


Check what the tables show. Do the missed downgrades sit at low scores? If they do, their inputs may have looked
healthy, and the downgrade may have come as a surprise. Do the false alarms look risky on paper (high debt, low
profit, weak rating)? If they do, S&P still did not act within a year. In both cases the model did what it can do
with public numbers on that day.

## A flag is a reason to look, not a reason to sell

Look again at the precision above. Is a large share of flagged reports **not** downgraded? A flag raises
the chance of a downgrade a lot (compare precision with the base rate, the share of all reports that were downgraded),
but it never makes the downgrade certain. And a downgrade by S&P is not the same as a falling share price or a
default.

So what should a flag lead to? **A closer look.** Read the company's latest 10-K and the rating agency's outlook,
check whether the bad numbers are one-off or lasting, and compare with similar companies. Then decide. A missing flag
does not mean "safe" either: about half of the downgrades were missed on the lecture's data. The `credit-watch` agent
in this repository does the first part of that look for you: it collects the ratings text from the filings and cites
it.

**What the lecture found on the real (licensed) data, for comparison.** Test years 2016-2024, 6,380 annual reports, of
which 862 were followed by an S&P downgrade within 12 months (13.5%).

| Result (real data) | Value |
|---|---|
| Logit: ROC-AUC / PR-AUC | 0.802 / 0.494 |
| Other single engines (not fly models): ROC-AUC | 0.785 to 0.808 (boosting 0.808; none clearly better than logit) |
| Averages with the logit: ROC-AUC | 0.817 (logit-boosting), 0.813 (all non-fly); both above the logit |
| S&P rating state alone: ROC-AUC | 0.757 |
| Fly brain models: ROC-AUC (notebook 4) | 0.759 to 0.807 (real wiring had no consistent edge over shuffled wiring) |
| Flag line (top 10% of an earlier year's scores) | caught 407 of 862 downgrades, with 8.7% false alarms |


## Tidy up

This notebook did not make a temporary folder, and it does not delete anything. The real table and the results stay in
`OUT_DIR`; the downloads stay in `DATA_DIR`.

## Check yourself

1. For test year 2020, from which year's scores is the flag line taken, and why not from 2020 itself?

<details><summary>Answer</summary>

From 2018, scored by a model trained on years up to 2016. Taking the line from 2020's own scores would need all of
2020's scores before flagging any company, which you do not have in real time.
</details>

2. On the lecture's data, about 29% of the 2020 reports were downgraded, but the model's average probability for 2020
   was about 14%. Does that mean the model ranked companies badly?

<details><summary>Answer</summary>

Not necessarily. ROC-AUC measures the ranking, and a ranking can stay good when the whole level is off. In a shock
year the probabilities are too low on average (here about half of the actual rate), so read them as relative risk, and
expect fewer companies than usual to cross a line fixed from an earlier year: in 2020 the line flagged only 46 of 695
reports (6.6%) and caught 39 of the 203 downgrades. (The missed share over all test years, 455 of the 862 downgrades,
is the other half of the picture.)
</details>

3. A company is flagged. Which is the better next step: sell, or look closer? Give one reason.

<details><summary>Answer</summary>

Look closer. Even flagged reports are often not followed by a downgrade (false alarms), and a downgrade is not the
same as a loss. The flag tells you where to spend your attention.
</details>